<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-HUYTRUONG/AIO-2026/blob/main/M3/ML-Base/Tree-based%20Algorithms/helpers/dataset_helper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Helper

Notebook này cung cấp các hàm tiện ích để lấy dataset CSV
cho các notebook Machine Learning.

## Supported environments

`dataset_helper` hỗ trợ hai môi trường chính:

### 1. Google Colab

Người dùng sẽ được yêu cầu upload file CSV trực tiếp.

### 2. Local / Jupyter / JupyterLab / VS Code

Helper sẽ:

1. Kiểm tra dataset đã tồn tại local hay chưa.
2. Nếu đã tồn tại → sử dụng lại dataset.
3. Nếu chưa tồn tại → download dataset từ GitHub Public Repository.

## Design goals

- Đơn giản.
- Không hard-code GitHub Token.
- Không cần authentication.
- Có local cache để tránh download nhiều lần.
- Các notebook Machine Learning không cần biết dataset được lấy từ đâu.

## Workflow

```text
                    get_csv_path()
                           │
                           ▼
                  Google Colab?
                    /          \
                  YES            NO
                   │              │
                   ▼              ▼
              Upload CSV      Local cache?
                                  │
                           ┌──────┴──────┐
                           │             │
                          YES            NO
                           │             │
                           ▼             ▼
                       Use local     GitHub Public
                                         │
                                         ▼
                                    Download CSV

## 1. Import libraries

Các thư viện được sử dụng:

- `pathlib.Path`: quản lý đường dẫn file.
- `urllib.request`: gửi HTTP request tới GitHub.
- `urllib.error`: xử lý lỗi HTTP và network.

In [4]:
from __future__ import annotations

from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

## 2. Dataset Configuration

Các configuration được tập trung tại một nơi để dễ thay đổi.

### Dataset local

`DEFAULT_DATASET_PATH` là tên file dataset khi lưu trên máy local.

### GitHub repository

Các biến:

- `GITHUB_OWNER`: GitHub username hoặc organization.
- `GITHUB_REPOSITORY`: tên repository.
- `GITHUB_BRANCH`: branch chứa dataset.
- `GITHUB_DATASET_PATH`: đường dẫn đến dataset bên trong repository.

Ví dụ repository:

```text
AIVIETNAM-AIO-HUYTRUONG/AIO-2026
```
Dataset:

```text
M3/ML-Base/Tree-based Algorithms/data/Student_Pass_Dataset.csv
```

In [5]:
DEFAULT_DATASET_PATH = "Student_Pass_Dataset.csv"

GITHUB_OWNER = "AIVIETNAM-AIO-HUYTRUONG"
GITHUB_REPOSITORY = "AIO-2026"
GITHUB_BRANCH = "main"

GITHUB_DATASET_PATH = (
    "M3/ML-Base/Tree-based%20Algorithms/"
    "data/Student_Pass_Dataset.csv"
)

GITHUB_RAW_URL = (
    f"https://raw.githubusercontent.com/"
    f"{GITHUB_OWNER}/"
    f"{GITHUB_REPOSITORY}/"
    f"{GITHUB_BRANCH}/"
    f"M3/ML-Base/Tree-based%20Algorithms/"
    f"data/{DEFAULT_DATASET_PATH}"
)

print(f"GITHUB_RAW_URL: {GITHUB_RAW_URL}")

GITHUB_RAW_URL: https://raw.githubusercontent.com/AIVIETNAM-AIO-HUYTRUONG/AIO-2026/main/M3/ML-Base/Tree-based%20Algorithms/data/Student_Pass_Dataset.csv


## 3. Detect Google Colab

Hàm `is_google_colab()` xác định notebook hiện tại
có đang chạy trên Google Colab hay không.

Kết quả:

```text
True  → Google Colab
False → Local / Jupyter / VS Code / môi trường khác

In [6]:
def is_google_colab() -> bool:
    """
    Kiểm tra notebook có đang chạy trên Google Colab hay không.

    Returns:
        True nếu đang chạy trên Google Colab.
        False nếu đang chạy ở môi trường khác.
    """

    try:
        import google.colab  # type: ignore

        return True

    except ImportError:
        return False

is_google_colab()

True

## 4. Validate local dataset

Trước khi download dataset từ GitHub, helper sẽ kiểm tra
dataset local đã tồn tại hay chưa.

Một file được xem là hợp lệ khi:

- File tồn tại.
- Đây là file thông thường.
- File không có kích thước bằng 0.

Việc kiểm tra local cache giúp tránh download dataset
mỗi lần notebook được chạy.

In [7]:
def is_valid_local_file(file_path: str) -> bool:
    """
    Kiểm tra file local có tồn tại và không bị rỗng.

    Args:
        file_path:
            Đường dẫn tới file cần kiểm tra.

    Returns:
        True nếu file tồn tại và có dữ liệu.
        False nếu file không tồn tại hoặc bị rỗng.
    """

    path = Path(file_path)

    return (
        path.exists()
        and path.is_file()
        and path.stat().st_size > 0
    )

## 5. Download dataset từ GitHub

Dataset nằm trong GitHub Public Repository nên không cần
GitHub Token.

Helper sử dụng GitHub Raw URL:

```text
GitHub Repository
        ↓
Raw URL
        ↓
HTTP Request
        ↓
CSV data
        ↓
Local file

In [8]:
def download_from_github(
    url: str,
    output_path: str,
) -> str:
    """
    Download dataset từ GitHub Public Repository.

    Args:
        url:
            GitHub Raw URL.

        output_path:
            Đường dẫn lưu dataset local.

    Returns:
        Đường dẫn tới dataset sau khi download.

    Raises:
        RuntimeError:
            Khi download thất bại.
    """

    print("Đang tải dataset từ GitHub...")
    print(f"URL: {url}")

    request = Request(
        url,
        headers={
            "User-Agent": "dataset-helper",
        },
    )

    try:
        with urlopen(
            request,
            timeout=30,
        ) as response:

            data = response.read()

    except HTTPError as error:
        raise RuntimeError(
            f"GitHub HTTP error "
            f"{error.code}: {error.reason}"
        ) from error

    except URLError as error:
        raise RuntimeError(
            f"Không thể kết nối tới GitHub: "
            f"{error.reason}"
        ) from error

    if not data:
        raise RuntimeError(
            "Dataset tải xuống bị rỗng."
        )

    path = Path(output_path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_bytes(data)

    print(
        f"Download thành công: {path}"
    )

    return str(path)

## 6. Upload dataset trên Google Colab

Khi chạy trên Google Colab, helper không download dataset từ GitHub.

Thay vào đó, người dùng sẽ được hiển thị file picker
để chọn file CSV từ máy tính.

Workflow:

```text
Google Colab
      ↓
files.upload()
      ↓
User chọn CSV
      ↓
Validate extension
      ↓
Return CSV path

In [9]:
def upload_csv_from_colab() -> str:
    """
    Upload CSV file từ Google Colab.

    Returns:
        Đường dẫn tới file CSV đã upload.
    """

    from google.colab import files

    print("Vui lòng chọn file CSV để upload...")

    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError(
            "Không tìm thấy file CSV. "
            "Vui lòng upload lại."
        )

    csv_path = next(iter(uploaded))

    if not csv_path.lower().endswith(".csv"):
        raise ValueError(
            f"File '{csv_path}' không phải là file CSV."
        )

    print(
        f"Upload thành công: {csv_path}"
    )

    return csv_path

## 7. Main function: `get_csv_path()`

Đây là function chính mà các notebook Machine Learning
sẽ sử dụng.

- Các notebook khác chỉ cần:

  ```python
  csv_path = get_csv_path()
  ```

- Google Colab

  ```text
  Colab
    ↓
  Upload CSV
  ```

- Local
  ```text
  Local
    ↓
  Dataset tồn tại?
    ├── YES → sử dụng dataset hiện tại
    │
    └── NO → download từ GitHub

In [10]:
def get_csv_path(
    default_path: str = DEFAULT_DATASET_PATH,
    github_raw_url: str = GITHUB_RAW_URL,
    force_download: bool = False,
) -> str:
    """
    Tự động lấy đường dẫn dataset CSV.

    Google Colab:
        Upload CSV thủ công.

    Non-Colab:
        1. Kiểm tra local cache.
        2. Nếu chưa có dataset:
           download từ GitHub Public Repository.

    Args:
        default_path:
            Đường dẫn lưu dataset local.

        github_raw_url:
            GitHub Raw URL của dataset.

        force_download:
            True:
                Luôn download dataset mới.

            False:
                Sử dụng dataset local nếu tồn tại.

    Returns:
        Đường dẫn tới dataset CSV.
    """

    # 1. Google Colab

    if is_google_colab():

        print("Môi trường: Google Colab")

        return upload_csv_from_colab()

    # 2. Local / Jupyter / VS Code

    print(
        "Môi trường: "
        "Jupyter / JupyterLab / VS Code / Local"
    )

    # 3. Local cache

    if (is_valid_local_file(default_path) and not force_download):

        print(f"Sử dụng dataset local: {default_path}")
        return default_path

    # 4. Download from GitHub

    print("Dataset local chưa tồn tại.")

    print("Đang download dataset từ GitHub...")

    return download_from_github(
        url=github_raw_url,
        output_path=default_path,
    )

## 8. Test `get_csv_path()`

Chạy cell này để kiểm tra toàn bộ workflow.

Function sẽ trả về đường dẫn dataset:

```python
csv_path = get_csv_path()

In [ ]:
# csv_path = get_csv_path()

# print()
# print("=" * 60)
# print(f"Dataset path: {csv_path}")
# print("=" * 60)